# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [42]:
print("""
Unit of analysis + time window

One row represents one content item for a client. The starter dataset contains
search and content performance information for each content item.

The available starter data represents a snapshot rather than a daily time-series
panel, so I will not claim a specific date range unless it is verified from the data.
""")


Unit of analysis + time window

One row represents one content item for a client. The starter dataset contains
search and content performance information for each content item.

The available starter data represents a snapshot rather than a daily time-series
panel, so I will not claim a specific date range unless it is verified from the data.



## 1. Unit of analysis + time window

One row represents one content item for one client on one reporting date.

For my SEO and content-performance lane, I use the daily performance table and focus on March 2026 (`month = '2026-03'`) for verification.

March contains daily observations, so the same content item and client can appear multiple times within the month.

For modeling, I will use historical observations available before the prediction moment and use a later period for the outcome. I will not use the final June 2026 month to develop label logic because it should be treated as a sealed test month.

The decision I want to support is identifying content items that show signs of declining search performance.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [43]:
feature_fields = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "position_volatility"
]

label_fields = [
    "is_declining"
]

context_fields = [
    "content_id",
    "client_id",
    "month"
]

excluded_fields = [
    "future_performance"
]

print("FEATURES:")
print(feature_fields)

print("\nLABEL:")
print(label_fields)

print("\nCONTEXT:")
print(context_fields)

print("\nEXCLUDED:")
print(excluded_fields)

print("\nReason for exclusion:")
print(
    "Future performance is excluded because it is not available "
    "at the decision moment and could cause target leakage."
)

FEATURES:
['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'position_volatility']

LABEL:
['is_declining']

CONTEXT:
['content_id', 'client_id', 'month']

EXCLUDED:
['future_performance']

Reason for exclusion:
Future performance is excluded because it is not available at the decision moment and could cause target leakage.


### Feature availability

**imp_prev30** — Knowable at the decision moment because it is calculated from impressions observed during the previous 30 days.

**visible_queries** — Knowable at the decision moment because it is calculated from search-query information already observed.

**rare_share** — Knowable at the decision moment because it is calculated from the historical distribution of observed queries.

**anon_share** — Knowable at the decision moment because it describes information already observed in the historical data.

**position_volatility** — Knowable at the decision moment because it summarizes search-position movement observed during the historical window.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [44]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created successfully.")

DuckDB connection created successfully.


In [45]:
# Show available tables/views in the DuckDB connection

print(con.execute("SHOW ALL TABLES").fetchdf())

Empty DataFrame
Columns: [database, schema, name, column_names, column_types, temporary]
Index: []


In [46]:
import pandas as pd
import duckdb

file_path = "/content/flyrank-starter/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(file_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [47]:
con.register("content_data", df)

print(con.execute("SHOW TABLES").fetchdf())

           name
0  content_data


In [48]:
test = con.execute("""
SELECT
    COUNT(*) AS total_rows
FROM content_data
""").fetchdf()

display(test)

,total_rows
0,30000


In [49]:
!grep -Rni "parquet\|huggingface\|hf://\|HF_TOKEN\|warehouse" /content/flyrank-starter 2>/dev/null | head -50

/content/flyrank-starter/requirements.txt:7:huggingface_hub>=0.24
/content/flyrank-starter/GUIDE.md:20:| `notebooks/03` | Weeks 3+: the **full warehouse release** via DuckDB + Hugging Face | The workflow your lane and capstone run on — aggregate in SQL, model in sklearn |
/content/flyrank-starter/GUIDE.md:29:| `requirements.txt` | pandas, numpy, scikit-learn, matplotlib, reportlab, duckdb, huggingface_hub | `pip install -r requirements.txt` |
/content/flyrank-starter/GUIDE.md:101:   %pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
/content/flyrank-starter/GUIDE.md:153:**Can I put the mentor-provided warehouse release in this repo?**
/content/flyrank-starter/.github/workflows/data-path-smoke.yml:4:# can never silently break. Needs the HF_TOKEN repo secret (read-only token);
/content/flyrank-starter/.github/workflows/data-path-smoke.yml:23:      HF_TOKEN: ${{ secrets.HF_TOKEN }}
/content/flyrank-starter/.github/workflows/data-path-smoke.yml:32:          if [ -z "$HF_

In [50]:
!find /content/flyrank-starter -type f | head -50

/content/flyrank-starter/scripts/02_baseline_score.py
/content/flyrank-starter/scripts/01_prepare_features.py
/content/flyrank-starter/scripts/ml_utils.py
/content/flyrank-starter/scripts/04_evaluate_and_export.py
/content/flyrank-starter/scripts/run_all.py
/content/flyrank-starter/scripts/05_build_pdf_report.py
/content/flyrank-starter/scripts/03_train_model.py
/content/flyrank-starter/requirements.txt
/content/flyrank-starter/data/raw/content_refresh_anonymized.csv
/content/flyrank-starter/GUIDE.md
/content/flyrank-starter/work/capstone_report_template.md
/content/flyrank-starter/work/README.md
/content/flyrank-starter/work/notebooks/w03_feature_leakage_check.ipynb
/content/flyrank-starter/work/notebooks/w01_research_question.ipynb
/content/flyrank-starter/work/notebooks/w02_ml_task_framing.ipynb
/content/flyrank-starter/work/notebooks/capstone.ipynb
/content/flyrank-starter/work/notebooks/w03_data_contract.ipynb
/content/flyrank-starter/work/notebooks/w06_validation_audit.ipynb
/con

In [51]:
!grep -Rni "parquet\|huggingface\|hf://\|HF_TOKEN\|warehouse" /content/flyrank-starter 2>/dev/null | head -50

/content/flyrank-starter/requirements.txt:7:huggingface_hub>=0.24
/content/flyrank-starter/GUIDE.md:20:| `notebooks/03` | Weeks 3+: the **full warehouse release** via DuckDB + Hugging Face | The workflow your lane and capstone run on — aggregate in SQL, model in sklearn |
/content/flyrank-starter/GUIDE.md:29:| `requirements.txt` | pandas, numpy, scikit-learn, matplotlib, reportlab, duckdb, huggingface_hub | `pip install -r requirements.txt` |
/content/flyrank-starter/GUIDE.md:101:   %pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
/content/flyrank-starter/GUIDE.md:153:**Can I put the mentor-provided warehouse release in this repo?**
/content/flyrank-starter/.github/workflows/data-path-smoke.yml:4:# can never silently break. Needs the HF_TOKEN repo secret (read-only token);
/content/flyrank-starter/.github/workflows/data-path-smoke.yml:23:      HF_TOKEN: ${{ secrets.HF_TOKEN }}
/content/flyrank-starter/.github/workflows/data-path-smoke.yml:32:          if [ -z "$HF_

In [52]:
!find /content/flyrank-starter -type f | head -50

/content/flyrank-starter/scripts/02_baseline_score.py
/content/flyrank-starter/scripts/01_prepare_features.py
/content/flyrank-starter/scripts/ml_utils.py
/content/flyrank-starter/scripts/04_evaluate_and_export.py
/content/flyrank-starter/scripts/run_all.py
/content/flyrank-starter/scripts/05_build_pdf_report.py
/content/flyrank-starter/scripts/03_train_model.py
/content/flyrank-starter/requirements.txt
/content/flyrank-starter/data/raw/content_refresh_anonymized.csv
/content/flyrank-starter/GUIDE.md
/content/flyrank-starter/work/capstone_report_template.md
/content/flyrank-starter/work/README.md
/content/flyrank-starter/work/notebooks/w03_feature_leakage_check.ipynb
/content/flyrank-starter/work/notebooks/w01_research_question.ipynb
/content/flyrank-starter/work/notebooks/w02_ml_task_framing.ipynb
/content/flyrank-starter/work/notebooks/capstone.ipynb
/content/flyrank-starter/work/notebooks/w03_data_contract.ipynb
/content/flyrank-starter/work/notebooks/w06_validation_audit.ipynb
/con

In [53]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Secret found:", HF_TOKEN is not None)
print("Token prefix:", HF_TOKEN[:3] if HF_TOKEN else "NONE")
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

Secret found: True
Token prefix: hf_
Token length: 37


In [54]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("DuckDB configured for FlyRank warehouse.")

DuckDB configured for FlyRank warehouse.


In [55]:
import duckdb

con = duckdb.connect()

con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"

CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Warehouse connection configured.")
print("Feature window: February 2026")
print("Label window: March 2026")

Warehouse connection configured.
Feature window: February 2026
Label window: March 2026


In [56]:
from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")

info = whoami(token=HF_TOKEN)

print("Hugging Face username:", info["name"])

Hugging Face username: sohamduchal12


In [57]:
from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")

print("Secret loaded:", bool(HF_TOKEN))

info = whoami(token=HF_TOKEN)

print("Hugging Face account:", info["name"])

Secret loaded: True
Hugging Face account: sohamduchal12


In [58]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Secret loaded:", bool(HF_TOKEN))
print("Starts with hf_:", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)

Secret loaded: True
Starts with hf_: True


In [59]:
test = con.execute(f"""
SELECT *
FROM {MAR}
LIMIT 5
""").fetchdf()

display(test)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [60]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected successfully.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected successfully.
Feature window: February 2026
Label window: March 2026


In [61]:
test = con.execute(f"""
SELECT *
FROM {MAR}
LIMIT 5
""").fetchdf()

display(test)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [62]:
query_1 = f"""
SELECT
    content_hash_id,
    client_hash_id,
    month,
    COUNT(*) AS row_count
FROM {MAR}
WHERE month = '2026-03'
GROUP BY content_hash_id, client_hash_id, month
ORDER BY row_count DESC
LIMIT 10
"""

result_1 = con.execute(query_1).fetchdf()
display(result_1)

,content_hash_id,client_hash_id,month,row_count
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03,31
1,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03,31
2,content_36c36abc7650d7af,client_73cda7b4e4f265ea,2026-03,31
3,content_a7da352b73b02668,client_73cda7b4e4f265ea,2026-03,31
4,content_05434271b257bb68,client_73cda7b4e4f265ea,2026-03,31
5,content_f39be42b42a4e8f6,client_73cda7b4e4f265ea,2026-03,31
6,content_1855a661b4d36130,client_73cda7b4e4f265ea,2026-03,31
7,content_22610b0934f8825e,client_73cda7b4e4f265ea,2026-03,31
8,content_712c365258cee05c,client_73cda7b4e4f265ea,2026-03,31
9,content_aafb2ab7e5fc80d0,client_73cda7b4e4f265ea,2026-03,31


### Interpretation

The result checks whether a content item/client/month combination appears once. If the returned `row_count` is 1, this supports my stated unit of analysis.


In [63]:
query_2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MAR}
WHERE month = '2026-03'
"""

result_2 = con.execute(query_2).fetchdf()
display(result_2)

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### Interpretation

This query gives the number of rows in my March 2026 slice and the observed date range. I use the measured result rather than assuming a date range.


In [64]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git /content/flyrank-starter

fatal: destination path '/content/flyrank-starter' already exists and is not an empty directory.


In [65]:
query_3 = f"""
SELECT
    COUNT(*) AS available_rows
FROM {MAR}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
"""

result_3 = con.execute(query_3).fetchdf()
display(result_3)

,available_rows
0,3611061


### Interpretation

I used `IS TRUE` to count rows where the availability condition is explicitly true. This avoids treating NULL or unknown availability as confirmed availability.


In [66]:
columns = con.execute(f"""
SELECT *
FROM {MAR}
LIMIT 0
""").description

for col in columns:
    print(col[0], "->", col[1])

report_date -> Date
client_hash_id -> STRING
content_hash_id -> STRING
client_has_gsc -> bool
client_has_ga4 -> bool
gsc_data_available -> bool
ga4_data_available -> bool
gsc_impressions -> NUMBER
gsc_clicks -> NUMBER
gsc_sum_position -> NUMBER
gsc_avg_position -> NUMBER
ga4_pageviews -> NUMBER
ga4_sessions -> NUMBER
ga4_users -> NUMBER
ga4_engaged_sessions -> NUMBER
ga4_total_engagement_sec -> NUMBER
sessions_organic -> NUMBER
sessions_direct -> NUMBER
sessions_referral -> NUMBER
sessions_social -> NUMBER
sessions_paid -> NUMBER
sessions_ai -> NUMBER
ai_chatgpt -> NUMBER
ai_perplexity -> NUMBER
ai_gemini -> NUMBER
ai_copilot -> NUMBER
ai_claude -> NUMBER
ai_meta -> NUMBER
ai_other -> NUMBER
scroll_events -> NUMBER
month -> STRING


In [67]:
sample = con.execute(f"""
SELECT *
FROM {MAR}
LIMIT 3
""").fetchdf()

display(sample)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [68]:
sample = con.execute(f"""
SELECT *
FROM {MAR}
LIMIT 3
""").fetchdf()

print("Columns available in March 2026:")
print(sample.columns.tolist())

display(sample)

Columns available in March 2026:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [69]:
# ML-04 — Five-feature frame
# Development month: March 2026

feature_query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    sessions_ai
FROM {MAR}
WHERE month = '2026-03'
"""

feature_frame = con.execute(feature_query).fetchdf()

print("Feature frame created successfully")
print("Rows:", len(feature_frame))
print("Columns:", len(feature_frame.columns))

display(feature_frame.head())

Feature frame created successfully
Rows: 9841378
Columns: 8


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,<NA>,<NA>


In [70]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]

print("Missing values before filling:")
print(feature_frame[feature_columns].isna().sum())

for col in feature_columns:
    feature_frame[col] = feature_frame[col].fillna(0)

print("\nMissing values after filling:")
print(feature_frame[feature_columns].isna().sum())

Missing values before filling:
gsc_impressions           0
gsc_clicks                0
gsc_avg_position    6230317
ga4_sessions        3018741
sessions_ai         3018741
dtype: int64

Missing values after filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
sessions_ai         0
dtype: int64


In [71]:
# Deliberate leakage experiment

leaky_frame = feature_frame.copy()

# Create a temporary label-like column.
# This is intentionally added to demonstrate leakage.
leaky_frame["label_leak"] = (
    leaky_frame["gsc_clicks"] < leaky_frame["gsc_impressions"]
).astype(int)

print("Leaky feature added:")
print("label_leak")

print("\nLeaky frame shape:", leaky_frame.shape)
print("Missing values:", leaky_frame.isna().sum().sum())

Leaky feature added:
label_leak

Leaky frame shape: (9841378, 9)
Missing values: 0


In [72]:
# Check missing values in the feature frame

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]

print("Missing values:")
print(feature_frame[feature_columns].isna().sum())


Missing values:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
sessions_ai         0
dtype: int64


In [73]:
# Fill missing values in the actual feature frame

feature_frame = feature_frame.copy()

for col in feature_columns:
    feature_frame[col] = feature_frame[col].fillna(0)

print("Missing values after filling:")
print(feature_frame[feature_columns].isna().sum())

Missing values after filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
sessions_ai         0
dtype: int64


In [74]:
# Deliberate leakage experiment

leaky_frame = feature_frame.copy()

leaky_frame["label_leak"] = (
    leaky_frame["gsc_clicks"] < leaky_frame["gsc_impressions"]
).astype(int)

print("Leaky feature added:", "label_leak")
print("Leaky frame shape:", leaky_frame.shape)
print("Missing values:", leaky_frame.isna().sum().sum())

Leaky feature added: label_leak
Leaky frame shape: (9841378, 9)
Missing values: 0


In [75]:
# Build the five-feature frame for March 2026

feature_query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    sessions_ai
FROM {MAR}
WHERE month = '2026-03'
"""

feature_frame = con.execute(feature_query).fetchdf()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

Feature frame shape: (9841378, 8)


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,<NA>,<NA>


In [76]:
# Deliberate leakage experiment

leaky_frame = feature_frame.copy()

# Create a temporary label for the leakage demonstration.
# This is intentionally added to the features to demonstrate leakage.
leaky_frame["label_leak"] = (
    leaky_frame["gsc_clicks"] < leaky_frame["gsc_impressions"]
).astype(int)

print("Leaky feature added:")
print("label_leak")

print("\nFeature columns:")
print(leaky_frame.columns.tolist())

Leaky feature added:
label_leak

Feature columns:
['content_hash_id', 'client_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'sessions_ai', 'label_leak']


In [77]:
APR = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

print("April 2026 source configured.")

April 2026 source configured.


In [78]:
label_query = f"""
WITH march AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {MAR}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, client_hash_id
),
april AS (
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM {APR}
    WHERE month = '2026-04'
    GROUP BY content_hash_id, client_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.march_impressions,
    a.april_impressions,
    CASE
        WHEN a.april_impressions < m.march_impressions
        THEN 1
        ELSE 0
    END AS is_declining
FROM march m
LEFT JOIN april a
    ON m.content_hash_id = a.content_hash_id
    AND m.client_hash_id = a.client_hash_id
"""

march_data = con.execute(label_query).fetchdf()

print("Label created.")
print("Rows:", len(march_data))
display(march_data.head())

Label created.
Rows: 331437


,content_hash_id,client_hash_id,march_impressions,april_impressions,is_declining
0,content_56a4dabd555eec90,client_62f4a7e64f5e0096,1.0,15.0,0
1,content_50266f97d6233542,client_62f4a7e64f5e0096,8.0,15.0,0
2,content_dc2422b7fc475fd6,client_62f4a7e64f5e0096,0.0,0.0,0
3,content_89edcdb8887fe6db,client_62f4a7e64f5e0096,0.0,0.0,0
4,content_d95e1739253d6a65,client_62f4a7e64f5e0096,208.0,168.0,1


In [79]:
print("is_declining distribution:")
print(march_data["is_declining"].value_counts(dropna=False))

print("\nMissing labels:")
print(march_data["is_declining"].isna().sum())

is_declining distribution:
is_declining
0    219470
1    111967
Name: count, dtype: int64

Missing labels:
0


In [80]:
# Aggregate March features to content + client level

feature_agg = con.execute(f"""
SELECT
    content_hash_id,
    client_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(sessions_ai) AS sessions_ai
FROM {MAR}
WHERE month = '2026-03'
GROUP BY content_hash_id, client_hash_id
""").fetchdf()

print("Aggregated feature frame:", feature_agg.shape)

display(feature_agg.head())

Aggregated feature frame: (331437, 7)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.147402,NaN,NaN
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,4.909314,NaN,NaN
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.074107,NaN,NaN
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,NaN,NaN
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,NaN,NaN


In [81]:
model_data = feature_agg.merge(
    march_data,
    on=["content_hash_id", "client_hash_id"],
    how="inner"
)

print("Model data shape:", model_data.shape)
print("Missing values:", model_data.isna().sum().sum())

display(model_data.head())

Model data shape: (331437, 10)
Missing values: 296100


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_ai,march_impressions,april_impressions,is_declining
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.147402,NaN,NaN,181.0,46.0,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,4.909314,NaN,NaN,34.0,12.0,1
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.074107,NaN,NaN,77.0,27.0,1
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,NaN,NaN,329.0,109.0,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,NaN,NaN,602.0,879.0,0


In [82]:
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]

for col in feature_columns:
    model_data[col] = model_data[col].fillna(0)

print("Missing feature values:")
print(model_data[feature_columns].isna().sum())

Missing feature values:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_sessions        0
sessions_ai         0
dtype: int64


In [83]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# -----------------------------
# DELIBERATE LEAKAGE TEST
# -----------------------------

X_leaky = model_data[feature_columns].copy()

# Intentionally add the target itself as a feature
X_leaky["label_leak"] = model_data["is_declining"]

y = model_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

print("LEAKY MODEL RESULTS")
print("-------------------")
print("Accuracy:", accuracy_score(y_test, leaky_pred))
print("F1:", f1_score(y_test, leaky_pred))

LEAKY MODEL RESULTS
-------------------
Accuracy: 1.0
F1: 1.0


In [84]:
X_leaky["label_leak"] = model_data["is_declining"]

In [85]:
# -----------------------------
# HONEST MODEL — LEAK REMOVED
# -----------------------------

# Use only the five legitimate features
X_honest = model_data[feature_columns].copy()

y = model_data["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

print("HONEST MODEL RESULTS")
print("--------------------")
print("Accuracy:", accuracy_score(y_test, honest_pred))
print("F1:", f1_score(y_test, honest_pred))

HONEST MODEL RESULTS
--------------------
Accuracy: 0.788027999034516
F1: 0.7095803432653196


### Leakage conclusion

I deliberately added the target-derived `label_leak` feature to demonstrate leakage. Because the feature directly contained the target, the model achieved an unrealistically high score.

I then removed `label_leak` and trained the model using only the five legitimate historical features. The second result is the honest score and is the result I retain.

This experiment shows why future information and target-derived fields must not be included in a predictive feature vector.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

1. The March warehouse contains daily observations, so the same content can appear on multiple dates.
2. GSC and GA4 availability is not consistent across all rows, so some traffic and ranking features contain missing values.
3. The label is based on observed future performance and does not prove that a feature caused a decline.
4. External factors such as Google algorithm changes, seasonality, competitors, and content changes are not fully represented.
5. The model is a decision-support tool and cannot guarantee that a content item will decline in the future.
6. The final month should be treated as a sealed outcome period rather than being used to develop the label logic.


In [86]:
print("""
Data limits

1. The starter dataset is a snapshot, so it does not provide complete daily history for every content item.

2. Missing values are present, so some features may not be available for every content item.

3. The data cannot explain all external factors that affect search performance.

4. The analysis can identify observed and directional patterns, but it cannot prove causation.

5. The model cannot predict Google's future ranking decisions.

6. Care is needed to avoid using future information when creating features and labels.
""")


Data limits

1. The starter dataset is a snapshot, so it does not provide complete daily history for every content item.

2. Missing values are present, so some features may not be available for every content item.

3. The data cannot explain all external factors that affect search performance.

4. The analysis can identify observed and directional patterns, but it cannot prove causation.

5. The model cannot predict Google's future ranking decisions.

6. Care is needed to avoid using future information when creating features and labels.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.